# Using the assignment data

This notebook demonstrates how to work with the assignment training data.


### Locating the project root

Before we start we need to find the project root directory so that we know where to find the data files. This looks a bit complicated but you don't need to worry about it for the assignment itself because you won't be writing code inside a notebook. The function `find_project_root` looks for marker files or directories (like `.git` or `pyproject.toml`) to identify the root of the project. We then save that in the variable `PROJECT_ROOT` for later use.


In [ ]:
from pathlib import Path

def find_project_root(current: Path, markers=("pyproject.toml", ".venv", ".git")) -> Path:
    """Find the project root directory by looking for marker files or directories."""
    current = current.resolve()
    for parent in [current] + list(current.parents):
        for m in markers:
            if (parent / m).exists():
                return parent
    raise FileNotFoundError(f"None of the markers {markers} found above {current}")

# Start from the notebook directory:
PROJECT_ROOT = find_project_root(Path.cwd())
print(f"Project root found at: {PROJECT_ROOT}")


If the cell above is not printing the expected project root path, then you can manually set it by uncommenting the cell below and replacing the string with the correct path.

In [ ]:
# PROJECT_ROOT = Path("/path/to/your/project/root")


## Working with the FBANK features

The FBANK features are stored in the joblib files, `fbank_speed.train.joblib` and `fbank_tempo.train.joblib`, for the speed and tempo modified variants respectively.

These can be loaded using the `joblib` library as follows:

In [ ]:
import joblib

data_file = PROJECT_ROOT / "data" / "fbank_tempo.train.joblib"  # Using Pathlib for windows/linux compatibility
fbank_data = joblib.load(data_file)
print(fbank_data.keys())


The data is stored as a dictionary with three keys: `features`, `target` and `filenames`.

- `features`: This contains the FBANK feature vectors for each audio sample. Features is a 2D numpy array where each row corresponds to one audio sample and contains a flattened array of FBANK features.
- `target`: This contains the class label (a digit 0-4) for each audio
- `filenames`: This contains the original filenames of the audio samples.

To display the FBANK features for a given row we can reshape the flattened array back into its original 2D form (number of frames x number of FBANK coefficients) and visualize it using a heatmap. Here's an example of how to do this:


In [ ]:
import matplotlib.pyplot as plt

SAMPLE_INDEX = 0  # You can change this index to visualize different samples
N_CHANNELS = 64 # The FBANK features use 64 channels
CLASSES = ('VERY SLOW', 'SLOW', 'NORMAL', 'FAST', 'VERY FAST')

fbank = fbank_data["features"][SAMPLE_INDEX]
target = int(fbank_data["target"][SAMPLE_INDEX])
fbank_2d = fbank.reshape(N_CHANNELS, -1)
plt.imshow(fbank_2d, aspect='auto', origin='lower', cmap='gray_r')
plt.title(f"FBANK Features for Sample Index {SAMPLE_INDEX}, class = {CLASSES[target]}")


## Working with the signal features

The signal features are stored in the joblib files, `signal_speed.train.joblib` and `signal_tempo.train.joblib`, for the speed and tempo modified variants respectively.

These can be loaded using the `joblib` library in the same way as the FBANK features.


In [ ]:
import joblib

data_file = PROJECT_ROOT / "data" / "signal_tempo.train.joblib"
signal_data = joblib.load(data_file)
print(signal_data.keys())


The data dictionary has the same structure as before, with `features`, `target` and `filenames` keys. However, the features are now the raw audio samples. Each audio segment is 1 second in duration with a sampling rate of 16 kHz, so each feature vector will have a length of 16,000 samples.

We can show these as a waveform using a plotting library such as Matplotlib. Here's an example of how to do this:

In [ ]:
import matplotlib.pyplot as plt

SAMPLE_INDEX = 1  # You can change this index to visualize different samples
CLASSES = ('VERY SLOW', 'SLOW', 'NORMAL', 'FAST', 'VERY FAST')

signal = signal_data["features"][SAMPLE_INDEX]
target = int(signal_data["target"][SAMPLE_INDEX])

plt.plot(signal)
plt.title(f"Signal for Sample Index {SAMPLE_INDEX}, class = {CLASSES[target]}")
plt.xlabel("Sample index")
plt.ylabel("Amplitude")

The signal can be played as audio in a Jupyter notebook using the `IPython.display.Audio` class


In [ ]:
from IPython.display import Audio
Audio(signal, rate=16000)  # assuming a sample rate of 16 kHz


## Jupyter Notebook Widgets demo

Jupyter notebooks support some interactive widgets that can be used to create simple user interfaces for exploring data. The following code creates a slider widget that allows you to select different sample indices and visualize the corresponding signals.

### Demo 1: Displaying FBANK Features

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import joblib

DATA_FILE = PROJECT_ROOT / "data" / "fbank_tempo.train.joblib"
fbank_data = joblib.load(DATA_FILE)    

N_SAMPLES = len(fbank_data["features"])
N_CHANNELS = 64
CLASSES = ('VERY SLOW', 'SLOW', 'NORMAL', 'FAST', 'VERY FAST')  


@interact(SAMPLE_INDEX=IntSlider(min=0, max=N_SAMPLES-1, step=1, value=0))
def plot_fbank(SAMPLE_INDEX):
    fbank = fbank_data["features"][SAMPLE_INDEX]
    target = int(fbank_data["target"][SAMPLE_INDEX])
    fbank_2d = fbank.reshape(N_CHANNELS, -1)
    plt.figure(figsize=(10, 4))
    plt.imshow(fbank_2d, aspect='auto', origin='lower', cmap='gray_r')
    plt.title(f"FBANK for Sample Index {SAMPLE_INDEX}, class = {CLASSES[target]}")
    plt.xlabel("Frame index")
    plt.ylabel("Filterbank channel")
    plt.colorbar()
    plt.show()




### Demo 2: Displaying Signal Features

Below we create a simple Jupyter notebook widget to explore the signal features interactively. You can use the slider to select different samples and visualize their waveforms. There is also an audio player to listen to the selected sample.


In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
from IPython.display import Audio, display
import joblib

DATA_FILE = PROJECT_ROOT / "data" / "signal_tempo.train.joblib"
signal_data = joblib.load(DATA_FILE)    

N_SAMPLES = len(signal_data["features"])
CLASSES = ('VERY SLOW', 'SLOW', 'NORMAL', 'FAST', 'VERY FAST')  


@interact(SAMPLE_INDEX=IntSlider(min=0, max=N_SAMPLES-1, step=1, value=0))
def plot_signal(SAMPLE_INDEX):
    signal = signal_data["features"][SAMPLE_INDEX]
    target = int(signal_data["target"][SAMPLE_INDEX])
    display(Audio(signal, rate=16000))  # explicitly display the audio widget
    plt.figure(figsize=(10, 4))
    plt.plot(signal)
    plt.title(f"Signal for Sample Index {SAMPLE_INDEX}, class = {CLASSES[target]}")
    plt.xlabel("Sample index")
    plt.ylabel("Amplitude")
    plt.show()


### Demo 3: Displaying FBANK and Signal Features

Below is a more advanced example showing both the signal and FBANK features and with a drop-down menu to allow you to select either the speed or tempo transform.

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown
from IPython.display import Audio
import joblib

DATA_FILE_STR = str(PROJECT_ROOT / "data" / "{feature}_{transform}.train.joblib")
N_CHANNELS = 64
CLASSES = ('VERY SLOW', 'SLOW', 'NORMAL', 'FAST', 'VERY FAST')  

# Load all the data
signal_data = {}
fbank_data = {}
for transform in ['tempo', 'speed']:
    signal_data[transform] = joblib.load(DATA_FILE_STR.format(feature="signal", transform=transform))    
    fbank_data[transform] = joblib.load(DATA_FILE_STR.format(feature="fbank", transform=transform))    

n_samples = len(signal_data["tempo"]["features"])

@interact(
    SAMPLE_INDEX=IntSlider(min=0, max=n_samples-1, step=1, value=0),
    transform=Dropdown(options=['tempo', 'speed'], value='tempo', description='Transform:')
)
def plot_all(SAMPLE_INDEX, transform):
    signal = signal_data[transform]["features"][SAMPLE_INDEX]
    target = int(signal_data[transform]["target"][SAMPLE_INDEX])
    fbank = fbank_data[transform]["features"][SAMPLE_INDEX]
    fbank_2d = fbank.reshape(N_CHANNELS, -1)

    display(Audio(signal, rate=16000))  # explicitly display the audio widget
    plt.figure(figsize=(10, 8))
    plt.subplot(2, 1, 1)
    plt.plot(signal)
    plt.title(f"Signal for Sample Index {SAMPLE_INDEX}, class = {CLASSES[target]}")
    plt.xlim(0, len(signal))
    plt.xlabel("Sample index")
    plt.ylabel("Amplitude")

    plt.subplot(2, 1, 2)
    plt.imshow(fbank_2d, aspect='auto', origin='lower', cmap='gray_r')
    plt.title(f"FBANK for Sample Index {SAMPLE_INDEX}, class = {CLASSES[target]}")
    plt.xlabel("Frame index")
    plt.ylabel("Filterbank channel")
 
    plt.tight_layout()
    plt.show()